In [2]:
!pip install rouge-score bert-score torch

In [1]:
!pip install ollama psutil nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 599.9 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.6 MB 599.9 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 657.8 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 657.8 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.6 MB 699.0 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 706.6 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 706.6 kB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 711.9 kB/s  0:00:02
   -------------------------------

In [2]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [1]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def evaluate_summary(generated_summary, reference_summary):

    if not generated_summary or not reference_summary:
        return {
            "ROUGE1": 0,
            "ROUGE2": 0,
            "ROUGEL": 0,
            "BERTScore": 0
        }

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = scorer.score(reference_summary, generated_summary)

    P, R, F1 = bertscore(
        [generated_summary],
        [reference_summary],
        lang="en",
        verbose=False
    )

    return {
        "ROUGE1": scores['rouge1'].fmeasure,
        "ROUGE2": scores['rouge2'].fmeasure,
        "ROUGEL": scores['rougeL'].fmeasure,
        "BERTScore": F1.mean().item()
    }

In [7]:
import time
import os
import psutil
import pandas as pd
import ollama
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure necessary NLP resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# 1. LOAD YOUR KAGGLE DATASET
csv_path = 'datasets/scisumm.csv' 

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Please make sure the dataset file is named '{csv_path}' and placed in the same directory as this notebook.")

df = pd.read_csv(csv_path)

# Preview the columns to ensure we map text correctly
print("Dataset columns:", df.columns.tolist())

# ScisummNet typically uses columns like 'document' or 'text' for the main paper body.
# We will identify the text column dynamically.
text_column = 'text' if 'text' in df.columns else df.columns[0]

# Sample 30 papers to keep the 1-week timeline realistic and fast
sample_df = df.head(10)

# 2. PROMPT COMPRESSION FUNCTION
def compress_prompt(text):
    if not isinstance(text, str):
        return ""
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text)
    compressed_tokens = [w for w in word_tokens if not w.lower() in stop_words]
    return " ".join(compressed_tokens)

# 3. EXPERIMENTAL TESTING ENGINE
results = []

def run_test(pipeline_name, model_name, input_text, reference_summary, paper_id):
    # Ensure text isn't excessively long for an edge model test loop (truncate to first 800 words)
    truncated_text = " ".join(str(input_text).split()[:800])
    
    process = psutil.Process()
    start_mem = process.memory_info().rss / (1024 * 1024) # MB
    start_time = time.time()
    
    try:
        # Requesting a summarization task from the local Ollama instance
        response = ollama.chat(model=model_name, messages=[
            {'role': 'user', 
            'content': f"Summarize this scientific text in two sentences: {truncated_text}"}
        ])
        output_text = response['message']['content']
    except Exception as e:
        output_text = f"Error during inference: {str(e)}"
        
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024 * 1024) # MB
    
    latency = end_time - start_time
    memory_used = max(0, end_mem - start_mem)

    metrics = evaluate_summary(
        output_text,
        reference_summary
    )

    
    return {
    'Paper_ID': paper_id,
    'Pipeline': pipeline_name,
    'Latency_Sec': round(latency, 3),
    'RAM_Used_MB': round(memory_used, 2),
    'Output_Word_Count': len(output_text.split()),
    "Compression_Ratio":
            len(output_text.split()) /
            max(1, len(truncated_text.split())),

    "ROUGE1": round(metrics["ROUGE1"],4),
    "ROUGE2": round(metrics["ROUGE2"],4),
    "ROUGEL": round(metrics["ROUGEL"],4),
    "BERTScore": round(metrics["BERTScore"],4),
    'Summary': output_text
}

# 4. RUN THE COMPARATIVE EXPERIMENT LOOP
print("\nStarting optimization benchmarks across 30 scientific papers...")

for idx, row in sample_df.iterrows():
    raw_text = row[text_column]
    reference_summary = row[summary_column]
    print(f"Processing Paper {idx + 1}/10...", end="\r")
    
    # Baseline Group: Full Text on Standard Model
    res_base = run_test("Baseline", "phi3", raw_text, reference_summary, idx)
    
    # Pipeline A Group: Full Text on Quantized Model
    res_pipe_a = run_test("Quantized_Model", "phi3:3.8b-mini-4k-instruct-q4_K_M", raw_text, reference_summary, idx)
    
    # Pipeline B Group: Stripped/Compressed Text on Standard Model
    compressed_text = compress_prompt(raw_text)
    res_pipe_b = run_test("Prompt_Compression", "phi3", compressed_text, reference_summary, idx)

    res_pipe_c = run_test(
    "Quantized_Prompt_Compression",
    "phi3:3.8b-mini-4k-instruct-q4_K_M",
    compressed_text,
    reference_summary,
    idx
)
    
    results.extend([res_base, res_pipe_a, res_pipe_b,res_pipe_c])

print("\nAll experiments complete!")


Dataset columns: ['text', 'summary']

Starting optimization benchmarks across 30 scientific papers...


NameError: name 'summary_column' is not defined

In [ ]:
# 5. CONVERT THE LOGGED METRICS INTO A DATAFRAME
results_df = pd.DataFrame(results)

# Rearrange the table so each Paper_ID is one row
pivot_df = results_df.pivot(
    index='Paper_ID',
    columns='Pipeline',
    values=['Latency_Sec', 'RAM_Used_MB', 'Output_Word_Count']
)

# Make Pipeline the top-level column and metrics the second-level column
pivot_df = pivot_df.swaplevel(0, 1, axis=1)

pipeline_order = [
    "Baseline",
    "Quantized_Model",
    "Prompt_Compression",
    "Quantized_Prompt_Compression"
]

metric_order = [
    "Latency_Sec",
    "RAM_Used_MB",
    "Output_Word_Count",
    "ROUGE1":"mean",
    "ROUGE2":"mean",
    "ROUGEL":"mean",
    "BERTScore":"mean"
]

pivot_df = pivot_df.reindex(
    columns=pd.MultiIndex.from_product([pipeline_order, metric_order])
)

# Calculate average for all numeric columns
average_row = pivot_df.mean()

# Add it as the last row
pivot_df.loc["Average"] = average_row


# Save to CSV
pivot_df.to_csv("llm_optimization_results.csv")

# Display first few rows
print(pivot_df.head())

In [ ]:
summary_df = results_df.pivot(
    index="Paper_ID",
    columns="Pipeline",
    values="Summary"
)

print(summary_df.head())

summary_df.to_csv("summaries.csv")